# Base model test for the query generator (planner)

Tests the model **before fine-tuning** on the same `test.jsonl`, with the same system prompt, the same generation settings and the same metrics as the fine-tuned model in `model.ipynb` (cells 45 to 50), so the two sets of numbers can be compared directly.

**Before you run it**
1. Stop the Docker containers (and any other notebook) that hold the GPU. The 9B model in 4 bit needs about 7 GB; the check in the third cell prints how much is free.
2. Restart the kernel, then run the cells from top to bottom. Nothing has to be installed if `model.ipynb` already ran in this environment (Unsloth, transformers, sentence-transformers).
3. Keep `test.jsonl` in the same folder as this notebook (it is written by cell 26 of `model.ipynb`). `train.jsonl` is needed only for the few-shot option.

**How to read the result.** A base model has not been taught the output format: it answers in markdown (topics, bullet lists, "Search Query: ..."), not in JSON. So there are **two scores**:
- **Strict score** (as in `model.ipynb`): the answer must be valid JSON. A markdown answer gets 0. This says whether the pipeline could use the model as it is.
- **Lenient score**: a reader turns the markdown into the same structure (topics, detail questions, search queries) and the same metrics are applied. This measures the *content* of the plan, independent of the format. It is a heuristic reader, so the notebook shows what it read; the raw answers are saved and the scores can be recomputed without generating again.

The similarity numbers are shown over valid (or readable) answers only, as `model.ipynb` does, and over all rows, where an unreadable answer counts as 0. The second one is the fair one when the rates differ a lot. For the report, give both scores and say which is which.

The run is **resumable**: every answer is written to a checkpoint file at once, and a rerun continues where it stopped.

In [ ]:
# ---- settings: change these, then run the notebook from top to bottom ----
BASE_MODEL_ID    = "Qwen/Qwen3.5-9B-Base"  # the model that was fine-tuned (model.ipynb, cell 10). For the post-trained model use "Qwen/Qwen3.5-9B"
LOAD_IN_4BIT     = True                     # the fine-tuned model was also tested in 4 bit (model.ipynb, cell 45)
MAX_SEQ_LENGTH   = 4096                     # the fine-tuned test used 2048; the room is needed only for the few-shot prompts
NUM_SHOTS        = 0                        # 0 = zero-shot (same prompt as the fine-tuned test); 2 = two worked examples from train.jsonl in the prompt
NUM_SAMPLES      = 50                       # first N rows of test.jsonl (shuffled with seed 42, so a random sample). The base model needs about 70 s per answer: 50 rows = about 1 hour. None = all rows
MAX_NEW_TOKENS   = 768                      # as generate_prediction in model.ipynb, cell 47
TEMPERATURE      = 0.3
TOP_P            = 0.9
DISABLE_THINKING = True                     # appends an empty <think></think> block to the prompt, as model.ipynb does
SEED             = 3407
TEST_FILE        = "test.jsonl"
TRAIN_FILE       = "train.jsonl"            # only used when NUM_SHOTS > 0
FINE_TUNED_RESULTS = "eval_results.json"    # saved by model.ipynb, cell 50; used for the comparison at the end (optional)

RUN_TAG = f"{BASE_MODEL_ID.split('/')[-1]}_{NUM_SHOTS}shot"
CHECKPOINT_PATH = f"eval_checkpoint_{RUN_TAG}.jsonl"
RESULTS_PATH    = f"eval_results_{RUN_TAG}.json"
print("run tag:", RUN_TAG)

In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"        # as in model.ipynb, cell 8

from unsloth import FastModel                  # unsloth first, before transformers
import json, re, time, random, difflib
import numpy as np
import torch
from tqdm.auto import tqdm

print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU found. Restart the kernel and check that the Docker containers are stopped."
free, total = torch.cuda.mem_get_info()
print(f"GPU: {torch.cuda.get_device_name(0)} | free {free / 1e9:.1f} GB of {total / 1e9:.1f} GB")
if free / 1e9 < 9:
    print("WARNING: less than 9 GB free. Stop the Docker containers or other notebooks that use the GPU, restart the kernel and run again.")

In [ ]:
# Load the base model (no LoRA adapter, no fine-tuning) the same way model.ipynb loads the fine-tuned one.
model, tokenizers = FastModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
)
FastModel.for_inference(model)
print("loaded", BASE_MODEL_ID, "| LoRA adapter present:", any("lora" in n.lower() for n, _ in model.named_parameters()))

In [ ]:
from datasets import load_dataset

SYSTEM_PROMPT = (
    "You are a research question decomposition agent. Given a research "
    "question, break it into topic-grouped sub-questions with detail "
    "questions, and provide search ready queries."
)

assert os.path.exists(TEST_FILE), f"{TEST_FILE} not found next to this notebook. It is written by cell 26 of model.ipynb."
test_dataset = load_dataset("json", data_files=TEST_FILE)["train"]
n_all = len(test_dataset)
if NUM_SAMPLES is not None:
    test_dataset = test_dataset.select(range(min(NUM_SAMPLES, n_all)))
print(f"test rows: {len(test_dataset)} of {n_all}")

# the file must be the one the fine-tuned model was tested on: chat format, the same system prompt
row = test_dataset[0]["messages"]
assert [m["role"] for m in row] == ["system", "user", "assistant"], "unexpected format: expected system, user, assistant messages"
assert row[0]["content"] == SYSTEM_PROMPT, "the system prompt in test.jsonl differs from the one used for the fine-tuned test"
print("question :", row[1]["content"])
print("reference:", row[2]["content"][:300], "...")

In [ ]:
# Few-shot examples (NUM_SHOTS > 0): worked examples from train.jsonl, short ones, chosen with a fixed seed.
def pick_shots(n, max_chars=1800):
    if n == 0:
        return []
    assert os.path.exists(TRAIN_FILE), f"{TRAIN_FILE} is needed for NUM_SHOTS > 0"
    train = load_dataset("json", data_files=TRAIN_FILE)["train"]
    order = list(range(len(train)))
    random.Random(SEED).shuffle(order)
    shots = []
    for i in order:
        msgs = train[i]["messages"]
        if len(msgs[2]["content"]) <= max_chars:
            shots.append((msgs[1]["content"], msgs[2]["content"]))
            if len(shots) == n:
                break
    return shots

SHOTS = pick_shots(NUM_SHOTS)


def build_messages(question):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for q, a in SHOTS:
        messages += [{"role": "user", "content": q}, {"role": "assistant", "content": a}]
    messages.append({"role": "user", "content": question})
    return messages


def generate_prediction(question, max_new_tokens=MAX_NEW_TOKENS):
    # Same as generate_prediction in model.ipynb (cell 47), plus the time and the number of new tokens.
    text = tokenizers.apply_chat_template(build_messages(question), tokenize=False, add_generation_prompt=True)
    if DISABLE_THINKING:
        text += "<think>\n\n</think>\n\n"
    inputs = tokenizers(text=text, return_tensors="pt").to("cuda")
    n_in = inputs["input_ids"].shape[1]
    t0 = time.time()
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        do_sample=True,
    )
    seconds = time.time() - t0
    new_tokens = int(outputs.shape[1] - n_in)
    raw = tokenizers.decode(outputs[0][n_in:], skip_special_tokens=True).strip()
    return raw, seconds, new_tokens


print(f"shots: {len(SHOTS)}")

In [ ]:
# Smoke test on 3 questions: look at what the base model actually writes before running the whole test.
torch.manual_seed(SEED)
for i in range(3):
    q = test_dataset[i]["messages"][1]["content"]
    raw, sec, ntok = generate_prediction(q)
    print(f"[{i}] {q}\n    {ntok} new tokens in {sec:.1f} s\n    {raw[:600]!r}\n")

In [ ]:
# ---- metrics: the same code as model.ipynb, cell 49 (the cosine uses numpy on normalised vectors, which gives the same value as scikit-learn) ----
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")


def extract_json_from_output(raw_output: str):
    raw_output = raw_output.strip()
    decoder = json.JSONDecoder()
    start = raw_output.find("{")
    if start == -1:
        return None
    try:
        obj, _ = decoder.raw_decode(raw_output[start:])
        return obj
    except json.JSONDecodeError:
        return None


def flatten_topics_only(parsed_json):
    if not parsed_json or "sub_questions" not in parsed_json:
        return []
    topics = []
    for item in parsed_json["sub_questions"]:
        if isinstance(item, dict) and "topic" in item:
            topics.append(item["topic"])
        elif isinstance(item, str):
            topics.append(item)
    return topics


def flatten_details_only(parsed_json):
    if not parsed_json or "sub_questions" not in parsed_json:
        return []
    all_details = []
    for item in parsed_json["sub_questions"]:
        if isinstance(item, dict):
            details = item.get("details", [])
            if isinstance(details, list):
                all_details.extend(str(d) for d in details)
    return all_details


def flatten_queries(parsed_json):
    if not parsed_json or "search_queries" not in parsed_json:
        return []
    return [q for q in parsed_json["search_queries"] if isinstance(q, str)]


def embedding_cosine_similarity(text_a: str, text_b: str) -> float:
    if not text_a.strip() or not text_b.strip():
        return 0.0
    e = embedder.encode([text_a, text_b], normalize_embeddings=True)
    return float(np.dot(e[0], e[1]))


def text_overlap_ratio(text_a: str, text_b: str) -> float:
    return difflib.SequenceMatcher(None, text_a.lower(), text_b.lower()).ratio()


def evaluate_single(question: str, generated_raw: str, reference_raw: str):
    gen_parsed = extract_json_from_output(generated_raw)
    ref_parsed = extract_json_from_output(reference_raw)

    result = {
        "question": question,
        "valid_json": gen_parsed is not None,
    }

    if gen_parsed is None or ref_parsed is None:
        result.update({
            "topic_similarity": 0.0,
            "details_similarity": 0.0,
            "query_similarity": 0.0,
            "topic_count_match": False,
            "query_count_match": False,
        })
        return result

    gen_topics = flatten_topics_only(gen_parsed)
    ref_topics = flatten_topics_only(ref_parsed)
    gen_details = flatten_details_only(gen_parsed)
    ref_details = flatten_details_only(ref_parsed)
    gen_queries = flatten_queries(gen_parsed)
    ref_queries = flatten_queries(ref_parsed)

    gen_topics_text = " ".join(gen_topics)
    ref_topics_text = " ".join(ref_topics)
    gen_details_text = " ".join(gen_details)
    ref_details_text = " ".join(ref_details)
    gen_queries_text = " ".join(gen_queries)
    ref_queries_text = " ".join(ref_queries)

    result.update({
        "num_sub_questions_generated": len(gen_topics),
        "num_sub_questions_reference": len(ref_topics),
        "num_details_generated": len(gen_details),
        "num_details_reference": len(ref_details),
        "num_search_queries_generated": len(gen_queries),
        "num_search_queries_reference": len(ref_queries),
        "topic_count_match": len(gen_topics) == len(ref_topics),
        "query_count_match": len(gen_queries) == len(ref_queries),
        "topic_similarity": embedding_cosine_similarity(gen_topics_text, ref_topics_text),
        "details_similarity": embedding_cosine_similarity(gen_details_text, ref_details_text),
        "query_similarity": embedding_cosine_similarity(gen_queries_text, ref_queries_text),
        "topic_text_overlap": text_overlap_ratio(gen_topics_text, ref_topics_text),
        "details_text_overlap": text_overlap_ratio(gen_details_text, ref_details_text),
        "query_text_overlap": text_overlap_ratio(gen_queries_text, ref_queries_text),
        "generated_topics": gen_topics,
        "reference_topics": ref_topics,
        "generated_details": gen_details,
        "reference_details": ref_details,
        "generated_queries": gen_queries,
        "reference_queries": ref_queries,
    })
    return result


def schema_ok(raw_output: str) -> bool:
    # Extra check for a base model: valid JSON is not enough, it must have both keys with non-empty lists inside.
    p = extract_json_from_output(raw_output)
    return (isinstance(p, dict) and isinstance(p.get("sub_questions"), list) and len(p["sub_questions"]) > 0
            and isinstance(p.get("search_queries"), list) and len(p["search_queries"]) > 0)

In [ ]:
# ---- lenient scoring: read the plan out of a markdown answer ----------------------------------------------------------------------
# The base model answers in markdown ("**Topic 1: ...**", bullet lists, "Search Query: ..."), not in the JSON the pipeline needs, so the strict score gives
# it 0. To still measure the CONTENT, this reader turns the markdown into the same structure {"sub_questions": [{"topic", "details"}], "search_queries": [...]}
# and the same metrics (evaluate_single above) are applied to it. It is a heuristic: the raw answers are saved, so it can be improved and the scores
# recomputed without generating again.
SECTION_RE = re.compile(r"^(research question|decomposed sub-?questions?|sub-?questions?|decomposition|search[- ]ready queries|search queries|queries|detail questions?|details?|notes?|summary|plan|answer)\s*:?\s*$", re.I)
TOPIC_RE   = re.compile(r"^(?:main\s+)?(?:topic(?:\s+group)?|sub-?question(?:\s+group)?|group|theme)\s*#?\s*\d*\s*[:.\-–—)]\s*(.+)$", re.I)
DETAIL_RE  = re.compile(r"^(?:detail(?:ed)?(?:\s+questions?)?|question)\s*#?\s*\d*\s*[:.\-–—)]\s*(.+)$", re.I)
QUERY_RE   = re.compile(r"^(?:search[- ]?(?:ready\s+)?quer(?:y|ies)|quer(?:y|ies))\s*#?\s*\d*\s*[:.\-–—)]\s*(.+)$", re.I)
BULLET_RE  = re.compile(r"^\s*(?:[*\-•+]|\d+[.)])\s+")


def _clean_line(line):
    t = line.strip()
    t = re.sub(r"^#+\s*", "", t)                                  # markdown heading marks
    t = re.sub(r"^(?:[*\-•+]\s+)+", "", t)                   # bullets
    t = t.replace("**", "").replace("__", "").replace("`", "").strip("* ")
    t = re.sub(r"^\(?\d+[.)]\s+", "", t)                          # numbering
    return re.sub(r"\s+", " ", t).strip()


def _unquote(t):
    return t.strip().strip("\"'“”‘’").strip()


def parse_markdown_plan(raw):
    topics, queries, mode = [], [], None
    for line in raw.splitlines():
        stripped = line.strip()
        if not stripped or set(stripped) <= set("-=_* "):         # an empty line or a horizontal rule
            continue
        is_heading = stripped.startswith("#")
        is_bullet = bool(BULLET_RE.match(line))
        whole_bold = bool(re.fullmatch(r"\*\*[^*].*[^*]\*\*:?", stripped)) and not is_bullet
        t = _clean_line(line)
        if not t or re.match(r"^research question\b", t, re.I):
            continue
        if SECTION_RE.fullmatch(t):                               # a header such as "Search Queries:" or "Detail Questions:"
            mode = "queries" if "quer" in t.lower() else ("details" if "detail" in t.lower() else None)
            continue
        m = QUERY_RE.match(t)
        if m:                                                     # "Search Query: ..."
            queries.append(_unquote(m.group(1)))
            continue
        m = TOPIC_RE.match(t)
        if m:                                                     # "Topic 1: ...", "Topic Group 2: ...", "Sub-question 3: ..."
            topics.append({"topic": _unquote(m.group(1)).rstrip(":"), "details": []})
            mode = "details"
            continue
        m = DETAIL_RE.match(t)
        if m and topics and mode != "queries":                    # "Detail Question: ..."
            topics[-1]["details"].append(_unquote(m.group(1)))
            continue
        if mode == "queries" and (is_bullet or not (whole_bold or is_heading)):
            queries.append(_unquote(t))                           # a line inside a "Search Queries" section
            continue
        if (whole_bold or is_heading) and not is_bullet:
            if t.endswith(":"):                                   # a bold label with nothing after it
                mode = "queries" if "quer" in t.lower() else mode
                continue
            topics.append({"topic": t, "details": []})            # a bold line is a topic: "**Mechanisms of Influence**"
            mode = "details"
            continue
        if is_bullet and topics and mode != "queries":            # a bullet under a topic is a detail question
            topics[-1]["details"].append(_unquote(t))
    topics = [t for t in topics if t["topic"]]
    queries = [q for q in queries if q]
    return {"sub_questions": topics, "search_queries": queries} if topics else None


def lenient_plan(raw):
    # a JSON answer is used as it is; otherwise the markdown is read. Returns (plan or None, "json" | "markdown" | "none")
    obj = extract_json_from_output(raw)
    if isinstance(obj, dict) and isinstance(obj.get("sub_questions"), list) and obj["sub_questions"]:
        return obj, "json"
    plan = parse_markdown_plan(raw)
    return (plan, "markdown") if plan else (None, "none")


def lenient_view(r):
    plan, source = lenient_plan(r["generated_raw"])
    ev = evaluate_single(r["question"], json.dumps(plan, ensure_ascii=False) if plan else "", r["reference_raw"])
    ev["source"] = source
    ev["schema_ok"] = bool(plan and plan.get("sub_questions") and plan.get("search_queries"))
    return ev


def attach_lenient(results):
    for r in results:
        r["lenient"] = lenient_view(r)
    return results


def as_view(results, which):
    if which == "strict":
        return results
    out = []
    for r in results:
        v = dict(r["lenient"])
        for k in ("seconds", "new_tokens"):
            if k in r:
                v[k] = r[k]
        out.append(v)
    return out


# ---- self-test of the reader on the three answer styles seen in the smoke test (and on JSON and on prose) ----
_A = ("**Research Question:** How did X?\n\n**Decomposed Sub-Questions:**\n\n**1. Topic: Mechanisms of Legislative Influence**\n"
      "*   What are the primary mechanisms (e.g., lobbying)?\n*   How do reformers translate opinion into law?\n\n**2. Topic: Case Studies**\n"
      "*   How did the temperance movement use legislation?\n\n**Search Queries:**\n*   \"legislation reformers\"\n*   \"temperance movement laws\"")
_B = ("**Research Question:** Y?\n\n**Topic Group 1: The Rise of Populism**\n*   **Detail Questions:**\n    *   How did hardship fuel discontent?\n"
      "    *   What policies did the Alliance advocate?\n\n**Topic Group 2: Party Strain**\n*   **Detail Questions:**\n    *   How did the split happen?\n\n"
      "**Search-Ready Queries:**\n1. texas democratic party populism\n2. farmers alliance texas")
_C = ("**Research Question:** Z?\n\n**Topic 1: Access & Convenience**\n*   Detail Question: How do paywalls impact access?\n*   Detail Question: What is your experience?\n"
      "*   Search Query: \"personal experience with paywalls\"\n*   Search Query: \"using library subscriptions\"")
_D = "```json\n{\"sub_questions\": [{\"topic\": \"t\", \"details\": [\"d\"]}], \"search_queries\": [\"q\"]}\n```"
for name, raw, topics, details, queries, src in [("A", _A, 2, [2, 1], 2, "markdown"), ("B", _B, 2, [2, 1], 2, "markdown"), ("C", _C, 1, [2], 2, "markdown"),
                                                  ("D", _D, 1, [1], 1, "json")]:
    plan, source = lenient_plan(raw)
    assert source == src and len(plan["sub_questions"]) == topics, (name, source, plan)
    assert [len(s["details"]) for s in plan["sub_questions"]] == details, (name, plan)
    assert len(plan["search_queries"]) == queries and not any('"' in q for q in plan["search_queries"]), (name, plan)
assert lenient_plan("Sure! Here is a plan: first define it, then compare.") == (None, "none")
print("reader self-test ok (3 markdown styles, JSON, prose)")

In [ ]:
def summarize(results, title="", show=True, kind="strict"):
    # Numbers as in model.ipynb (averages over the valid answers), plus the same averages over ALL rows (an invalid answer counts as 0)
    # and the share of answers that have the full structure. kind="lenient": "valid" means that a plan could be read from the answer.
    n_total = len(results)
    valid = [r for r in results if r["valid_json"]]
    n_valid = len(valid)
    keys = ["topic_similarity", "details_similarity", "query_similarity"]
    out = {
        "rows": n_total,
        "valid_json_rate": n_valid / n_total if n_total else 0.0,
        "schema_ok_rate": sum(1 for r in results if r.get("schema_ok", r["valid_json"])) / n_total if n_total else 0.0,
    }
    for k in keys:
        out[k + "_valid_only"] = sum(r[k] for r in valid) / n_valid if n_valid else 0.0
        out[k + "_all_rows"] = sum(r[k] for r in results) / n_total if n_total else 0.0
    out["topic_count_match_rate_valid_only"] = sum(r["topic_count_match"] for r in valid) / n_valid if n_valid else 0.0
    out["query_count_match_rate_valid_only"] = sum(r["query_count_match"] for r in valid) / n_valid if n_valid else 0.0
    out["topic_count_match_rate_all_rows"] = sum(r["topic_count_match"] for r in results) / n_total if n_total else 0.0
    out["query_count_match_rate_all_rows"] = sum(r["query_count_match"] for r in results) / n_total if n_total else 0.0
    secs = [r["seconds"] for r in results if "seconds" in r]
    toks = [r["new_tokens"] for r in results if "new_tokens" in r]
    if secs:
        out["avg_seconds_per_answer"] = sum(secs) / len(secs)
        out["avg_new_tokens"] = sum(toks) / len(toks)
        out["hit_token_limit_rate"] = sum(1 for t in toks if t >= MAX_NEW_TOKENS) / len(toks)
    if show:
        print("=" * 64)
        print(title or "Summary")
        print(f"Rows evaluated:                    {n_total}")
        lab1, lab2 = ("Valid JSON rate:", "Valid JSON with both keys:") if kind == "strict" else ("Plan could be read:", "Plan with topics and queries:")
        print(f"{lab1:35s}{n_valid}/{n_total} ({100 * out['valid_json_rate']:.1f}%)")
        print(f"{lab2:35s}{100 * out['schema_ok_rate']:.1f}%")
        print(f"Avg topic similarity:              {out['topic_similarity_valid_only']:.3f}  (valid only)   {out['topic_similarity_all_rows']:.3f}  (all rows)")
        print(f"Avg details similarity:            {out['details_similarity_valid_only']:.3f}  (valid only)   {out['details_similarity_all_rows']:.3f}  (all rows)")
        print(f"Avg query similarity:              {out['query_similarity_valid_only']:.3f}  (valid only)   {out['query_similarity_all_rows']:.3f}  (all rows)")
        print(f"Topic count match rate:            {100 * out['topic_count_match_rate_valid_only']:.1f}%  (valid only)   {100 * out['topic_count_match_rate_all_rows']:.1f}%  (all rows)")
        print(f"Query count match rate:            {100 * out['query_count_match_rate_valid_only']:.1f}%  (valid only)   {100 * out['query_count_match_rate_all_rows']:.1f}%  (all rows)")
        if secs:
            print(f"Avg time per answer:               {out['avg_seconds_per_answer']:.1f} s, {out['avg_new_tokens']:.0f} new tokens, {100 * out['hit_token_limit_rate']:.1f}% hit the {MAX_NEW_TOKENS}-token limit")
        print("=" * 64)
    return out


def run_evaluation(dataset, checkpoint_path=CHECKPOINT_PATH, resume=True, verbose=False):
    # As run_evaluation in model.ipynb (cell 49): one result per line is written to the checkpoint at once, and a rerun skips what is done.
    results, done = [], set()
    if resume and os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    r = json.loads(line)
                except json.JSONDecodeError:
                    continue                      # a half-written last line after a crash
                results.append(r)
                done.add(r["question"])
        if results:
            print(f"Resuming: {len(results)} rows already in {checkpoint_path}")
    torch.manual_seed(SEED)
    with open(checkpoint_path, "a", encoding="utf-8") as f:
        for i in tqdm(range(len(dataset)), desc="Evaluating"):
            row = dataset[i]
            question = row["messages"][1]["content"]
            reference_raw = row["messages"][2]["content"]
            if question in done:
                continue
            generated_raw, seconds, new_tokens = generate_prediction(question)
            r = evaluate_single(question, generated_raw, reference_raw)
            r.update({"schema_ok": schema_ok(generated_raw), "seconds": round(seconds, 2), "new_tokens": new_tokens,
                      "generated_raw": generated_raw, "reference_raw": reference_raw})
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
            f.flush()
            os.fsync(f.fileno())
            results.append(r)
            done.add(question)
            if verbose:
                print(f"[{i}] valid={r['valid_json']} schema_ok={r['schema_ok']} topic={r['topic_similarity']:.3f} query={r['query_similarity']:.3f}")
    return results

In [ ]:
# The test itself. A first pass of 300 rows is enough to see the picture; set NUM_SAMPLES = None in the settings cell (and run from the top) for all rows.
eval_results = run_evaluation(test_dataset)
base_summary = summarize(eval_results, title=f"STRICT SCORE (valid JSON only): {BASE_MODEL_ID}, {NUM_SHOTS}-shot")

In [ ]:
# Lenient score: read the plan out of every answer (JSON as it is, markdown with the reader above) and score it with the same metrics.
# Cheap: it needs no generation, only the saved answers, so it can be rerun after the reader is improved.
attach_lenient(eval_results)
lenient_summary = summarize(as_view(eval_results, "lenient"), title=f"LENIENT SCORE (plan read from the answer): {BASE_MODEL_ID}, {NUM_SHOTS}-shot", kind="lenient")

# Check what the reader made of the first answers: counts of topics and queries against the reference
for r in eval_results[:8]:
    l = r["lenient"]
    print(f"[{l['source']:8s}] topics {l.get('num_sub_questions_generated', 0)} (ref {l.get('num_sub_questions_reference', '-')}), "
          f"queries {l.get('num_search_queries_generated', 0)} (ref {l.get('num_search_queries_reference', '-')})  {r['question'][:60]}")

# one answer next to what was read from it
first = next((r for r in eval_results if r["lenient"]["source"] == "markdown"), None)
if first is not None:
    print("\n--- raw answer (start) ---\n" + first["generated_raw"][:700])
    print("\n--- what the reader took from it ---")
    print(json.dumps(lenient_plan(first["generated_raw"])[0], indent=1, ensure_ascii=False)[:1200])

In [ ]:
# Why do the invalid answers fail? (the part that explains the numbers)
from collections import Counter

def failure_kind(r):
    raw = r["generated_raw"]
    if r.get("schema_ok"):
        return "ok: valid JSON with both keys"
    if r["valid_json"]:
        return "valid JSON but wrong or missing keys"
    if "{" not in raw:
        return "no JSON at all (prose, a list, or empty)"
    return "JSON started but broken or cut off"

kinds = Counter(failure_kind(r) for r in eval_results)
for k, v in kinds.most_common():
    print(f"{v:5d}  {100 * v / len(eval_results):5.1f}%  {k}")
print("\nwhat the lenient reader found:", dict(Counter(r["lenient"]["source"] for r in eval_results)),
      "| answers cut at the token limit:", sum(1 for r in eval_results if r.get("new_tokens", 0) >= MAX_NEW_TOKENS))

shown = Counter()
print()
for r in eval_results:
    k = failure_kind(r)
    if k.startswith("ok") or shown[k] >= 2:
        continue
    shown[k] += 1
    print(f"--- {k}\nquestion: {r['question']}\nanswer  : {r['generated_raw'][:500]!r}\n")

In [ ]:
# Comparison with the fine-tuned model (model.ipynb, cell 50: 4,821 test rows, checkpoint-32538).
FINE_TUNED_FROM_NOTEBOOK = {   # the printed summary of model.ipynb, cell 50 (valid answers only)
    "rows": 4821, "valid_json_rate": 4801 / 4821, "topic_similarity_valid_only": 0.901, "details_similarity_valid_only": 0.838,
    "query_similarity_valid_only": 0.894, "topic_count_match_rate_valid_only": 0.452, "query_count_match_rate_valid_only": 0.106,
}
ft = None
if os.path.exists(FINE_TUNED_RESULTS):
    ft_all = json.load(open(FINE_TUNED_RESULTS, encoding="utf-8"))
    questions = {r["question"] for r in eval_results}
    ft_rows = [r for r in ft_all if r["question"] in questions]          # the same questions as the base model got
    if ft_rows:
        attach_lenient(ft_rows)                                           # its answers are JSON, so the lenient score is almost the strict one
        ft = summarize(ft_rows, show=False)
        ft_len = summarize(as_view(ft_rows, "lenient"), show=False, kind="lenient")
        print(f"fine-tuned model on the same {len(ft_rows)} questions (from {FINE_TUNED_RESULTS})")
if ft is None:
    ft = ft_len = FINE_TUNED_FROM_NOTEBOOK
    print(f"WARNING: {FINE_TUNED_RESULTS} not found or no common questions: using the printed numbers of model.ipynb over all 4,821 rows (valid answers only).")

def pct(x):
    return "n/a" if x is None else f"{100 * x:.1f}%"

def num(x):
    return "n/a" if x is None else f"{x:.3f}"

def row(name, key, fmt):
    # base model strict, base model lenient, fine-tuned model (its answers are JSON: the strict score)
    return (name, fmt(base_summary.get(key)), fmt(lenient_summary.get(key)), fmt(ft.get(key)))

rows = [
    row("Answer is valid JSON / a plan can be read",  "valid_json_rate",                       pct),
    row("Plan with topics and search queries",        "schema_ok_rate",                        pct),
    row("Topic similarity (answers read only)",       "topic_similarity_valid_only",           num),
    row("Details similarity (answers read only)",     "details_similarity_valid_only",         num),
    row("Query similarity (answers read only)",       "query_similarity_valid_only",           num),
    row("Topic similarity (all rows, unread = 0)",    "topic_similarity_all_rows",             num),
    row("Details similarity (all rows, unread = 0)",  "details_similarity_all_rows",           num),
    row("Query similarity (all rows, unread = 0)",    "query_similarity_all_rows",             num),
    row("Topic count match (answers read only)",      "topic_count_match_rate_valid_only",     pct),
    row("Query count match (answers read only)",      "query_count_match_rate_valid_only",     pct),
]
print(f"\n{'':46s}{'base STRICT':>13s}{'base LENIENT':>14s}{'fine-tuned':>13s}")
for name, b_strict, b_len, f_ in rows:
    print(f"{name:46s}{b_strict:>13s}{b_len:>14s}{f_:>13s}")
print("\nSTRICT: only valid JSON counts (can the pipeline use the model as it is). LENIENT: the plan is read from the answer, markdown included (how good is the content).")

In [ ]:
# Save everything and free the GPU.
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump({"settings": {"model": BASE_MODEL_ID, "shots": NUM_SHOTS, "samples": len(eval_results), "max_new_tokens": MAX_NEW_TOKENS,
                            "temperature": TEMPERATURE, "top_p": TOP_P, "load_in_4bit": LOAD_IN_4BIT, "seed": SEED},
               "summary_strict": base_summary, "summary_lenient": lenient_summary, "results": eval_results}, f, indent=2, ensure_ascii=False)
print("saved", RESULTS_PATH)

import gc
for name in ["model", "tokenizers", "embedder"]:
    if name in globals():
        del globals()[name]
for _ in range(3):
    gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still held by this kernel: {torch.cuda.memory_allocated() / 1e9:.2f} GB")